## **10_transformer_block: Assembling One Complete Transformer Block & The Full GPT Model**

This is the grand payoff.  
We will take `CausalSelfAttention` and `MLP` and assemble them using `Residual Connections` and `Layer Normalization` into one complete, powerful, stackable **Block**.

Then we'll stack those blocks to build the full **GPT-2 model**.

### The Block: Our Fundamental Lego Brick

```python
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x
```

There is **zero new complexity** here. It's just a container for the components we've already mastered.

### The Blueprint (`__init__`)

It's just an assembly list — like the parts list for a Lego set:

+ `self.ln_1` — the first stabilizer (LayerNorm), applied before attention
+ `self.attn` — the communication layer (CausalSelfAttention)
+ `self.ln_2` — the second stabilizer, applied before the MLP
+ `self.mlp` — the thinking layer (MLP)

### The Data's Journey (`forward`)

The `forward` method follows a simple, powerful mantra: **Normalize → Process → Add**

```
Input x ──→ LN1 → Attention ──→ (+) ──→ LN2 → MLP ──→ (+) ──→ Output
   │                              ↑         │                    ↑
   └──────── residual ────────────┘         └──── residual ──────┘
```

**Sub-layer 1 — The Attention Meeting:**
1. **Normalize:** `self.ln_1(x)` — stabilize the input
2. **Process:** `self.attn(...)` — tokens exchange information
3. **Add:** `x + ...` — residual connection

**Sub-layer 2 — The Thinking Time:**
1. **Normalize:** `self.ln_2(x)` — stabilize again
2. **Process:** `self.mlp(...)` — each token thinks independently
3. **Add:** `x + ...` — residual connection

The most critical property: **output shape = input shape** `(B, T, C)`. This makes it stackable.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

@dataclass
class GPTConfig:
    n_embd: int = 64
    n_head: int = 4
    block_size: int = 128
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.drop(self.proj(x))
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# Test the Block
config = GPTConfig()
block = Block(config)

x = torch.randn(1, 8, config.n_embd)
out = block(x)
print("Input shape: ", x.shape)
print("Output shape:", out.shape)
print("\nShapes match:", x.shape == out.shape, "→ Stackable!")

### Stacking Blocks: From a Single Meeting to a Symposium

A single Block is like a single project meeting.  
To build deep understanding, we need a **series** of these meetings.

+ **Meeting 1 (Block 0):** Figures out basic grammar and syntax
+ **Meeting 2 (Block 1):** Takes that and pieces together higher-level ideas
+ **Meeting 12 (Block 11):** Deep, nuanced, semantic understanding

| Concept | What it Does | Analogy | Why? |
| :--- | :--- | :--- | :--- |
| **Width** (Multi-Head) | Parallel processing *within* a layer | 12 specialists in **one meeting** | Many perspectives at the same level |
| **Depth** (Multi-Layer) | Sequential processing *across* layers | **12 meetings**, each refining the last | Hierarchical understanding |

### `nn.ModuleList`: Creating the Series of Meetings

```python
self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
```

**Are the weights shared between blocks?**  
**No.** Each `Block(config)` creates an independent object with its own unique weights.  
Layer 1 processes raw embeddings; layer 12 makes final refinements.  
Different layers need different skills.

In [ ]:
@dataclass
class FullGPTConfig:
    vocab_size: int = 100
    block_size: int = 32
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1

# Stack of blocks
config = FullGPTConfig()
blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])

print(f"Number of blocks: {len(blocks)}")
print(f"Each block is independent: {blocks[0].attn.c_attn is not blocks[1].attn.c_attn}")

# Flow data through the entire stack
x = torch.randn(1, 8, config.n_embd)
print(f"\nInput shape: {x.shape}")

for i, block in enumerate(blocks):
    x = block(x)
    print(f"After Block {i}: {x.shape}")

### The Full GPT-2 Model Architecture

```
Input Token IDs
    ↓
Token & Positional Embeddings (wte + wpe)
    ↓
Dropout
    ↓
N × Transformer Blocks
    ↓
Final Layer Norm (ln_f)
    ↓
Language Model Head (next chapter)
    ↓
Output Logits
```

In [ ]:
class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Part 1: The Input Layers
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

        # Part 2: The Core Processing Layers
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])

        # Part 3: The Output Layer
        self.ln_f = nn.LayerNorm(config.n_embd)

    def forward(self, idx):
        B, T = idx.size()

        # Embeddings
        tok_emb = self.wte(idx)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.wpe(pos)
        x = self.drop(tok_emb + pos_emb)

        # Transformer Blocks
        for block in self.h:
            x = block(x)

        # Final LayerNorm
        x = self.ln_f(x)
        return x

# Test the model (without lm_head for now)
config = FullGPTConfig()
model = GPT2(config)

idx = torch.randint(0, config.vocab_size, (2, 8))  # Batch of 2, length 8
output = model(idx)

print("Input (token IDs) shape:", idx.shape)
print("Output shape:", output.shape)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

We have now defined almost all the structural components of our model:  
the entrance, the main processing tower, and the final stabilization stage.

**Part 1** — `wte` + `wpe` + `drop`: Convert raw token IDs into meaningful vectors  
**Part 2** — `h`: The deep stack of Transformer Blocks  
**Part 3** — `ln_f`: Final stabilization before prediction

The only piece missing is the most important one for a language model:  
the layer that **actually makes the prediction** — the **Language Model Head**.